In [ ]:
import cassiopeia as cas
import cProfile
from collections import defaultdict
import copy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import time
from tqdm.auto import tqdm
import utilities
from scipy.interpolate import interp1d

In [ ]:
# =========================================================
#                    1. 全局参数设置
# =========================================================
num_simulations_fig2 = 100  # 图2：每个条件跑 100 次计算概率
num_batches_fig1 = 10       # 图1：跑 10 个批次，用于计算 95% 置信区间 (误差带)
depth = 8
num_states = 5
lamb = 0.5
q_dist = dict(zip(range(1, num_states + 1), [1 / num_states] * num_states))

# 扫描参数
k_cand_list = [10, 20, 50, 100, 150, 200]
p_miss_list = [0.0, 0.1, 0.2]
target_acc_list = np.linspace(0.6, 0.95, 8) # 图1的 X 轴：目标准确率

# =========================================================
#            2. 数据生成 - Figure 2 (成功概率)
# =========================================================
print("========== Generating Data for Figure 2 (Success Probability) ==========")
results_fig2 = []
acc_threshold = 0.9  # 定义“成功”的准确率阈值

for p_miss in p_miss_list:
    for k_cand in k_cand_list:
        success_count = 0
        for _ in tqdm(range(num_simulations_fig2), desc=f"k={k_cand}, p_miss={p_miss}"):
            # 1. 模拟真实树
            ground_truth_tree = utilities.complete_binary_tree_sim(k_cand, q_dist, lamb, depth)
            
            # TODO: 如果需要应用 p_miss，请取消下面这行的注释并确保您的 utilities 中有此函数
            # ground_truth_tree = utilities.apply_missing_data(ground_truth_tree, p_miss)
            
            # 2. OTO/MAX-Cut 重构
            triplets = utilities.find_recon_triplets(ground_truth_tree)
            recon_tree = utilities.build_tree_from_triplet_partition(ground_truth_tree, triplets)
            
            # 3. 计算准确率
            acc = utilities.calculate_triplets_correct(ground_truth_tree, recon_tree)
            
            if acc >= acc_threshold:
                success_count += 1
                
        # 计算该参数组合下的成功概率
        prob = success_count / num_simulations_fig2
        results_fig2.append({
            "p_miss": f"$p_{{miss}} = {p_miss}$", 
            "k_cand": k_cand, 
            "Success_Probability": prob
        })

df_fig2 = pd.DataFrame(results_fig2)
df_fig2.to_csv("fig2_success_probability.csv", index=False)
print("Data saved to fig2_success_probability.csv")

# =========================================================
#            3. 数据生成 - Figure 1 (理论 vs 模拟 K)
# =========================================================
print("\n========== Generating Data for Figure 1 (Required Capacity K) ==========")
# 辅助函数：通过扫描计算特定目标准确率所需的经验最小 k
def get_empirical_k_for_target(target_acc, p_miss_val=0.0):
    k_test_range = [10, 30, 50, 100, 200]
    acc_results = []
    for k in k_test_range:
        acc_vals = []
        for _ in range(5): # 跑少量次取平均
            gt_tree = utilities.complete_binary_tree_sim(k, q_dist, lamb, depth)
            # gt_tree = utilities.apply_missing_data(gt_tree, p_miss_val)
            trips = utilities.find_recon_triplets(gt_tree)
            r_tree = utilities.build_tree_from_triplet_partition(gt_tree, trips)
            acc_vals.append(utilities.calculate_triplets_correct(gt_tree, r_tree))
        acc_results.append(np.mean(acc_vals))
    
    # 线性插值找到达到 target_acc 所需的 K
    interp_func = interp1d(acc_results, k_test_range, kind='linear', fill_value="extrapolate")
    return float(interp_func(target_acc))

results_fig1 = []

for batch in tqdm(range(num_batches_fig1), desc="Running Batches for 95% CI"):
    for t_acc in target_acc_list:
        sim_k = get_empirical_k_for_target(t_acc, p_miss_val=0.0)
        results_fig1.append({
            "Target_Accuracy": t_acc,
            "Required_k": sim_k,
            "Type": "Simulated (MAX-Cut)"
        })

df_fig1 = pd.DataFrame(results_fig1)
df_fig1.to_csv("fig1_required_capacity.csv", index=False)
print("Data saved to fig1_required_capacity.csv")

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# =========================================================
#                    读取数据
# =========================================================
df_fig1 = pd.read_csv("fig1_required_capacity.csv")
df_fig2 = pd.read_csv("fig2_success_probability.csv")

# =========================================================
#                    理论界限计算函数 (仅作图用)
# =========================================================
depth = 8
def theoretical_k_bound(acc):
    """
    请替换为您论文中推导出的真实公式 
    这里仅为演示公式的平滑曲线性态
    """
    return max(5, 8 * np.log(2**depth) / ((1.01 - acc)**1.5))

# 从 DataFrame 中提取目标准确率的唯一值作为 X 轴列表
target_acc_list = sorted(df_fig1["Target_Accuracy"].unique())

# =========================================================
#                    可视化绘图
# =========================================================
sns.set_theme(style="whitegrid", rc={"axes.edgecolor": "black", "grid.color": "lightgrey"})
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# ----------------- Panel A: Theoretical vs Simulated K -----------------
ax0 = axes[0]

# 画出模拟数据的折线图 + 95% CI 误差带
sns.lineplot(
    data=df_fig1, 
    x="Target_Accuracy", 
    y="Required_k", 
    color="#FF8C00", # 选用近似橙色
    marker="o",
    errorbar=("ci", 95), # 自动计算 95% 置信区间并绘制阴影
    label="Simulated (MAX-Cut)",
    ax=ax0
)

# 计算并画出理论极限折线
theo_k_vals = [theoretical_k_bound(acc) for acc in target_acc_list]
ax0.plot(target_acc_list, theo_k_vals, color="#4682B4", linestyle="--", linewidth=2.5, label="Theoretical Limit (OTO)")

ax0.set_title("Validation of Required Recording Capacity", fontsize=14, fontweight='bold', pad=15)
ax0.set_xlabel("Target Accuracy ($Acc_{target}$)", fontsize=12)
ax0.set_ylabel("Required Capacity ($k_{req}$)", fontsize=12)
ax0.legend(loc="upper left", frameon=True, fontsize=10)

# ----------------- Panel B: Success Probability vs k -----------------
ax1 = axes[1]

# 使用 seaborn 根据 p_miss 画出多条折线
sns.lineplot(
    data=df_fig2, 
    x="k_cand", 
    y="Success_Probability", 
    hue="p_miss", 
    palette="viridis", # 采用从紫到绿的冷色渐变，科研风极强
    marker="s", 
    linewidth=2,
    ax=ax1
)

# 添加一条表示 90% 概率的辅助基准线
ax1.axhline(y=0.9, color='red', linestyle=':', linewidth=1.5, label="90% Reliability")

ax1.set_title("Global Tree Reconstruction Reliability", fontsize=14, fontweight='bold', pad=15)
ax1.set_xlabel("Number of Target Sites ($k_{cand}$)", fontsize=12)
ax1.set_ylabel("Success Probability ($Acc \geq 0.9$)", fontsize=12)
ax1.set_ylim(-0.05, 1.05)
ax1.legend(title="Missing Rate", loc="lower right", frameon=True, fontsize=10)

# ----------------- 细节与保存 -----------------
# 标注 Panel 字母 a, b
fig.text(0.04, 0.95, "a", fontsize=16, fontweight="bold")
fig.text(0.50, 0.95, "b", fontsize=16, fontweight="bold")

plt.tight_layout(rect=[0.05, 0, 1, 1])
plt.savefig("Fig_Capacity_Reliability.pdf", dpi=300, bbox_inches="tight")
plt.show()
print("Plots saved successfully as 'Fig_Capacity_Reliability.pdf'")